## Swinging Atwood Machine: Lösung hier mit RK4

(Siehe Lecture notes 1.3.3)


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Store generated files in data/01_mechanik/<notebook name>.
# Finding the repository root makes this work regardless of VS Code's kernel working directory.
notebook_name = "SwingingAtwood"
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / ".git").exists()),
    Path.cwd(),
)
output_dir = project_root / "data" / "01_mechanik" / notebook_name
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

In [ ]:
# State vector y = [phi , l2, dot_phi , dot_l2 ]
# Bewegungsgleichung ist d/dt y = f(y), mit
# d/dt phi     = dot_phi
# d/dt l2      = dot_l2
# d/dt dot_phi = - (2*dot_phi*dot_l2 + sin(phi)/l2
# d/dt dot_l2  = (mu * l2 *dot_phi**2 -1 + mu*cos(phi))/(1+mu)

# Beachte: Die REchnung luaft einfach weiter wenn l2<0 oder l2>1:
# die physikalische L¨sing ist dann nicht mehr gültig


def f_swm(y,mu):
    f=np.zeros(4)
    f[0]=y[2] 
    f[1]=y[3]
    f[2]=-(2*y[2]*y[3]+np.sin(y[0]))/y[1]
    f[3]=(mu*y[1]*y[2]**2-1+mu*np.cos(y[0]))/(1+mu)
    return f


def rk4_step_swm(y,h,mu):
    k1=f_swm(y,mu)
    k2=f_swm(y+h*0.5*k1,mu)
    k3=f_swm(y+h*0.5*k2,mu)
    k4=f_swm(y+h*k3,mu)
    return y + (h/6.0)*(k1+2*k2+2*k3+k4)

def rk4_full_swm(h,nmax,y0,mu):
    tpoints = np.zeros(nmax)
    ypoints=np.full((nmax, 4), 0.0)    
    yn=y0
    for n in range(nmax):
        tn=n*h
        tpoints[n]=tn
        yn=rk4_step_swm(yn,h,mu)
        ypoints[n]=yn.copy()
    return tpoints,ypoints



In [ ]:

y0=np.array([0.2,0.4,0,0])
nmax=1000
h=0.003
mu=0.3


ts1,ys1=rk4_full_swm(h,nmax,y0,mu)
ts2,ys2=rk4_full_swm(h*0.5,2*nmax,y0,mu)


# Calculate the values for plotting
x1_values = ys1[:, 1] * np.sin(ys1[:, 0])  # y values
y1_values = -ys1[:, 1] * np.cos(ys1[:, 0])  # x values
x1_values_1 = ys2[:, 1] * np.sin(ys2[:, 0])  # y values
y1_values_1 = -ys2[:, 1] * np.cos(ys2[:, 0])  # x values

# Create the plot
plt.figure(figsize=(10, 6))
#plt.plot(t1, ys[:,0], marker='o', linestyle='-', color='b')  # Use 'o' for points
#plt.plot(t1, ys[:,1], marker='+', linestyle='-', color='r')  # Use 'o' for points
plt.plot(x1_values, y1_values, '-', color='r')  # Use 'o' for points
#plt.plot(x1_values_1, y1_values_1, '--', color='k')  # Use 'o' for points
plt.title(f'$\mu$={mu:.3f}, $\phi$(0)={y0[0]:.3f}, $l_2$(0)={y0[1]:.3f}, $\dot\phi$(0)={y0[2]:.3f}, $\dot l_2$(0)={y0[3]:.3f}')
#plt.title(f'Plot of $\mu$={mu:.3f}, $\phi$(0)={y0[0]:.3f}')
plt.ylabel('$z_2$')
plt.grid()
plt.axhline(0, color='black', linewidth=0.5, ls='--')  # Add x-axis line
plt.axvline(0, color='black', linewidth=0.5, ls='--')  # Add y-axis line
plt.show()



In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import Video, display

# Example data (replace with your actual x1, y1, x2, y2, ts data)
x1 = ys1[:, 1] * np.sin(ys1[:, 0])  # y values
y1 = -ys1[:, 1] * np.cos(ys1[:, 0])  # x values
x2 = 0*ys1[:, 1]  # y values
y2 = ys1[:, 1]-1   # x values
ts=ts1


# Choose every p-th point from the series
p = 10
x1 = x1[::p]
y1 = y1[::p]
x2 = x2[::p]
y2 = y2[::p]
ts = ts[::p]

# Create a figure and axis with the required limits
fig, ax = plt.subplots()
ax.set_xlim(-1, 1)  # Restrict the x-axis to [-1, 1]
ax.set_ylim(-1, 0.1)  # Restrict the y-axis to [-1, 0.1]

# Initialize the connecting lines
line1, = ax.plot([0, x1[0]], [0, y1[0]], color='black')  # Line from origin to Circle 1
line2, = ax.plot([0, x2[0]], [0, y2[0]], color='black')  # Line from origin to Circle 2

# Create two circles (with smaller radius) and a text label for time
circle1 = plt.Circle((x1[0], y1[0]), 0.05*(mu**(1/3)), color='blue')  # Blue circle
circle2 = plt.Circle((x2[0], y2[0]), 0.05, color='red')   # Red circle
time_label = ax.text(-0.9, 0.05, '', fontsize=12)  # Text label for the time

# Add the static dashed circle around the origin
static_circle = plt.Circle((0, 0), 1, color='black', fill=False, linestyle='--')
ax.add_patch(static_circle)

# Add the moving circles to the plot
ax.add_patch(circle1)
ax.add_patch(circle2)


# Update function for the animation
def update(frame):
    
    # Update the time label
    time_label.set_text(f't = {ts[frame]:.2f}')
    
    # Update the connecting lines
    line1.set_data([0, x1[frame]], [0, y1[frame]])  # Line connecting to Circle 1
    line2.set_data([0, x2[frame]], [0, y2[frame]])  # Line connecting to Circle 2
    # Update positions of the circles
    circle1.set_center((x1[frame], y1[frame]))
    circle2.set_center((x2[frame], y2[frame]))
    
    return line1, line2, circle1, circle2, time_label

# Create the animation
ani = FuncAnimation(fig, update, frames=len(ts), interval=50, blit=True)

# Save the animation and embed it directly in the notebook.
# Muted videos may autoplay in modern browsers; playback controls remain available.
video_file = output_dir / 'swinging_circles.mp4'
ani.save(str(video_file), writer='ffmpeg', fps=20)
plt.close(fig)
display(Video(str(video_file), embed=True, html_attributes='controls autoplay loop muted'))
